In [10]:
import pandas as pd
import numpy as np
import pennylane as qml
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import root_mean_squared_error, r2_score
from qiskit.circuit.library import pauli_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

In [11]:
dataset = pd.read_csv("../dataset/riemann_features.csv")

X = dataset.drop(columns=["distance"])
y = dataset["distance"]

X = X[:1000]
y = y[:1000]

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [12]:
random_forest_features = pd.read_csv("../results/experiment_11/random_forest_feature_selection.csv")["feature"].tolist()
correlation_features = pd.read_csv("../results/experiment_11/correlation_feature_selection.csv")["feature"].tolist()
gevrey_method_features = pd.read_csv("../results/experiment_11/gevrey_method_feature_selection.csv")["feature"].tolist()
mrmr_10_features = pd.read_csv("../results/experiment_11/mrmr_10_features.csv")["feature"].tolist()
mi_features = pd.read_csv("../results/experiment_11/mi_feature_selection.csv")["feature"].tolist()
rank_aggregation_features = pd.read_csv("../results/data_analysis/selected_features.csv")["feature"].tolist()

features_map = {
    "Random Forest": random_forest_features[:10],
    "Correlation": correlation_features[:10],
    "Gevrey Method": gevrey_method_features[:10],
    "MRMR (10 features)": mrmr_10_features,
    "Mutual Information": mi_features,
    "Rank Aggregation": rank_aggregation_features
}

# 2) Training Classical SVR

In [13]:
def run_classical_experiment(features_map):
    results = []

    param_grid = {
        "svr__C": [0.1, 1, 5, 10],
        "svr__epsilon": [0.001, 0.01, 0.1, 0.5],
        "svr__gamma": ["scale", 0.01, 0.1, 1.0]
    }

    for group_name, features in features_map.items():
        print(f"Running experiment for group: {group_name} with {len(features)} features")

        X_train_subset = X_train[features].to_numpy()
        X_test_subset = X_test[features].to_numpy()

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(kernel="rbf"))
        ])

        grid = GridSearchCV(
            pipeline,
            param_grid,
            scoring="neg_root_mean_squared_error",
            cv=TimeSeriesSplit(n_splits=5),
            n_jobs=-1
        )

        grid.fit(X_train_subset, y_train)

        best_model = grid.best_estimator_

        y_pred = best_model.predict(X_test_subset)

        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results.append({
            "Group": group_name,
            "RMSE": rmse,
            "R2": r2,
            "Best C": grid.best_params_["svr__C"],
            "Best epsilon": grid.best_params_["svr__epsilon"],
            "Best gamma": grid.best_params_["svr__gamma"],
            "Features": len(features)
        })

    return pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)


In [14]:
df_standard = run_classical_experiment(features_map)
print(df_standard)
df_standard.to_csv("../results/experiment_11/classical_results.csv", index=False)

Running experiment for group: Random Forest with 10 features
Running experiment for group: Correlation with 10 features
Running experiment for group: Gevrey Method with 10 features
Running experiment for group: MRMR (10 features) with 10 features
Running experiment for group: Mutual Information with 10 features
Running experiment for group: Rank Aggregation with 10 features
                Group      RMSE        R2  Best C  Best epsilon Best gamma  \
0    Rank Aggregation  0.071669  0.941874       5          0.01        0.1   
1  Mutual Information  0.095787  0.896169       1          0.01      scale   
2  MRMR (10 features)  0.112232  0.857458       1          0.01      scale   
3       Random Forest  0.187287  0.603062       5          0.10      scale   
4       Gevrey Method  0.197501  0.558584       5          0.01       0.01   
5         Correlation  0.262881  0.217962       1          0.10       0.01   

   Features  
0        10  
1        10  
2        10  
3        10  
4     

# 2) Traning Quantum SVR using the same best hyperparameters and features of classical SVR using Angle Embedding

## Naive implementation, only rotations in X and standard scaling

In [15]:
# from pennylane.kernels import target_alignment

# alignment = target_alignment(X_tr, y_train, kernel, assume_normalized_kernel=True)
# print(f"Kernel alignment: {alignment:.4f}")

In [16]:
results = []

param_grid = {
    "svr__C": [1, 5],
    "svr__epsilon": [0.01, 0.1]
}

N_QUBITS = 10

In [17]:
dev = qml.device("default.qubit", wires=10)

@qml.qnode(dev)
def kernel(x1, x2, n_qubits=10):
    qml.AngleEmbedding(x1, wires=range(n_qubits), rotation="X")
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits), rotation="X")
    return qml.expval(qml.Projector([0] * n_qubits, wires=range(n_qubits)))


def kernel_mat(A, B):
    return qml.kernels.kernel_matrix(A, B, kernel)


features = features_map["Rank Aggregation"]

print(f"Running quantum experiment for group: Rank Aggregation with {len(features)} features")

X_train_subset = X_train[features].to_numpy()
X_test_subset = X_test[features].to_numpy()

pipeline = Pipeline([
    ("scaler", MinMaxScaler(feature_range=(0, np.pi))),
    ("svr", SVR(kernel=kernel_mat))
])

grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=TimeSeriesSplit(n_splits=5),
    n_jobs=-1
)

grid.fit(X_train_subset, y_train)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test_subset)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

results.append({
    "Experiment": "Quantum SVR Angle Embedding Kernel (X rotation)",
    "RMSE": rmse,
    "R2": r2,
    "Best C": grid.best_params_["svr__C"],
    "Best epsilon": grid.best_params_["svr__epsilon"],
    "qubits": N_QUBITS,
})

print(pd.DataFrame(results))

Running quantum experiment for group: Rank Aggregation with 10 features
                                        Experiment      RMSE        R2  \
0  Quantum SVR Angle Embedding Kernel (X rotation)  0.088547  0.911273   

   Best C  Best epsilon  qubits  
0       5          0.01      10  


## Rotations in X, Y, and Z and entanglement

In [18]:
dev = qml.device("default.qubit", wires=N_QUBITS)

def ansatz_layer(x, wires):
    for _ in range(2):
        qml.AngleEmbedding(x, wires=wires, rotation="X")
        qml.BasicEntanglerLayers(
            weights=np.zeros((1, len(wires))),
            wires=wires,
            rotation=qml.RZ
        )
        qml.AngleEmbedding(x, wires=wires, rotation="Y")


@qml.qnode(dev)
def kernel(x1, x2):
    ansatz_layer(x1, wires=range(N_QUBITS))
    qml.adjoint(ansatz_layer)(x2, wires=range(N_QUBITS))
    return qml.expval(qml.Projector([0] * N_QUBITS, wires=range(N_QUBITS)))


def kernel_mat(A, B):
    return qml.kernels.kernel_matrix(A, B, kernel)


X_train_subset = X_train[features].to_numpy()
X_test_subset = X_test[features].to_numpy()

pipeline = Pipeline([
    ("scaler", MinMaxScaler(feature_range=(0, np.pi))),
    ("svr", SVR(kernel=kernel_mat))
])

grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=TimeSeriesSplit(n_splits=5),
    n_jobs=-1
)

grid.fit(X_train_subset, y_train)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test_subset)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

results.append({
    "Experiment": "Quantum SVR Angle Embedding Kernel (X, Y, Z rotations with entanglement)",
    "RMSE": rmse,
    "R2": r2,
    "Best C": grid.best_params_["svr__C"],
    "Best epsilon": grid.best_params_["svr__epsilon"],
    "qubits": N_QUBITS,
})

print(pd.DataFrame(results))

                                          Experiment      RMSE        R2  \
0    Quantum SVR Angle Embedding Kernel (X rotation)  0.088547  0.911273   
1  Quantum SVR Angle Embedding Kernel (X, Y, Z ro...  0.114931  0.850519   

   Best C  Best epsilon  qubits  
0       5          0.01      10  
1       5          0.01      10  


In [ ]:
dev = qml.device("default.qubit", wires=N_QUBITS)

def ansatz_layer(x, wires):
    qml.AngleEmbedding(x, wires=wires, rotation="X")
    qml.BasicEntanglerLayers(
        weights=np.zeros((1, len(wires))),
        wires=wires,
        rotation=qml.RZ
    )
    qml.AngleEmbedding(x, wires=wires, rotation="Y")


@qml.qnode(dev)
def kernel(x1, x2):
    ansatz_layer(x1, wires=range(N_QUBITS))
    qml.adjoint(ansatz_layer)(x2, wires=range(N_QUBITS))
    return qml.expval(qml.Projector([0] * N_QUBITS, wires=range(N_QUBITS)))


def kernel_mat(A, B):
    return qml.kernels.kernel_matrix(A, B, kernel)


X_train_subset = X_train[features].to_numpy()
X_test_subset = X_test[features].to_numpy()

pipeline = Pipeline([
    ("scaler", MinMaxScaler(feature_range=(0, np.pi))),
    ("svr", SVR(kernel=kernel_mat))
])

grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=TimeSeriesSplit(n_splits=5),
    n_jobs=-1
)

grid.fit(X_train_subset, y_train)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test_subset)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

results.append({
    "Experiment": "Quantum SVR Angle Embedding Kernel (Entanglement, X and Y rotations)",
    "RMSE": rmse,
    "R2": r2,
    "Best C": grid.best_params_["svr__C"],
    "Best epsilon": grid.best_params_["svr__epsilon"],
    "qubits": N_QUBITS,
})

print(pd.DataFrame(results))

## Traning Quantum SVR using the same best hyperparameters and features of classical SVR using PauliFeatureMap

In [19]:
def create_quantum_kernel(n_qubits, reps=3, entanglement="full", paulis=["ZZ", "Z"]):
    feature_map = pauli_feature_map(feature_dimension=n_qubits, reps=reps, entanglement=entanglement, paulis=paulis)
    sampler = StatevectorSampler()
    fidelity = ComputeUncompute(sampler=sampler)
    quantum_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)
    return quantum_kernel

In [20]:
print("Running quantum experiments with Pauli Feature Map\n")

scaler = StandardScaler()
X_train_subset = X_train[features].to_numpy()
X_test_subset = X_test[features].to_numpy()

quantum_kernel = create_quantum_kernel(n_qubits=N_QUBITS)

X_train_subset = X_train[features].to_numpy()
X_test_subset = X_test[features].to_numpy()

pipeline = Pipeline([
    ("scaler", MinMaxScaler(feature_range=(0, np.pi))),
    ("svr", SVR(kernel=quantum_kernel.evaluate))
])

grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=TimeSeriesSplit(n_splits=5),
    n_jobs=-1
)

grid.fit(X_train_subset, y_train)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test_subset)

rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

results.append({
    "Experiment": "Quantum SVR Pauli Feature Map Kernel",
    "RMSE": rmse,
    "R2": r2,
    "Best C": grid.best_params_["svr__C"],
    "Best epsilon": grid.best_params_["svr__epsilon"],
    "qubits": N_QUBITS,
})

Running quantum experiments with Pauli Feature Map



KeyboardInterrupt: 

In [ ]:
quantum_df = pd.DataFrame(results)
print(quantum_df)
quantum_df.to_csv("../results/experiment_11/quantum_results.csv", index=False)